# Generating a Stratified Haunted Places Subset for Solr Ingestion (Religon-Based Sampling)

This script creates a stratified random subset of the haunted places dataset to prepare for full-text and numeric querying in Apache Solr.
It first loads the full dataset, drops non-essential columns (like image captions and object detections), and cleans numeric fields such as Total_Deaths, Average_Mental_Health_Days, and Depression_Prevalence to ensure consistent formats.

For the sampling strategy, the script performs stratified sampling based on Religion_Intersection, selecting at least three haunted places per religion group (or all if fewer than three exist). It then fills the rest of the subset with additional random rows to reach a total of approximately 1,000 entries.

The final dataset retains broader metadata — including health statistics, apparition descriptions, and location details — making it suitable for complex Solr queries across both text and numeric fields.

The output is saved as a line-delimited JSON file (haunted_places_subset_stratified_solr.json), formatted specifically for Solr ingestion.
This subset was generated separately from the radial chart subset to support the different objectives of Task 3: enabling interactive search and exploration of cultural and environmental patterns in haunting reports via Solr.

In [2]:
import pandas as pd
import json
import numpy as np

# Load dataset
df = pd.read_csv("../data/processed/haunted_places_features_added_v2.tab", sep="\t")

# Drop problematic columns
drop_cols = ["Image_Caption", "Image_Objects", "Locations"]
df = df.drop(columns=drop_cols, errors="ignore")

# Clean numeric fields
df["Total_Deaths"] = pd.to_numeric(df["Total_Deaths"], errors="coerce").fillna(0).astype(int)
df["Average_Mental_Health_Days"] = pd.to_numeric(df["Average_Mental_Health_Days"], errors="coerce").fillna(0)
df["Average_Poor_Health_Days"] = pd.to_numeric(df["Average_Poor_Health_Days"], errors="coerce").fillna(0)
df["Depression_Prevalence"] = pd.to_numeric(df["Depression_Prevalence"], errors="coerce").fillna(0)

# STRATIFIED SAMPLING by Religion_Intersection
# Minimum 3 per religion (or all if <3)
grouped = df.groupby("Religion_Intersection", group_keys=False)
stratified_sample = grouped.apply(lambda g: g.sample(n=min(3, len(g)), random_state=42))

# Combine with extra random rows to reach 1000 if needed
remaining = 1000 - len(stratified_sample)
if remaining > 0:
    random_sample = df.drop(stratified_sample.index).sample(n=remaining, random_state=42)
    final_df = pd.concat([stratified_sample, random_sample])
else:
    final_df = stratified_sample

# Export to line-delimited JSON for Solr ingestion
output_path = "../data/processed/haunted_places_subset_stratified_solr.json"
with open(output_path, "w") as f:
    for record in final_df.to_dict(orient="records"):
        json.dump(record, f)
        f.write("\n")